# Fine-tune Mammogram Model CBIS-DDSM


In [ ]:
# ── Download models from Google Drive ────────────────────────────────────────
!pip install -U --no-cache-dir gdown -q

import gdown

gdown.download('https://drive.google.com/uc?id=1hK3UiacZ4Onucudh9E0frVg3iz35hwNn',
 '/kaggle/working/mammo_model.h5', quiet=False)
gdown.download('https://drive.google.com/uc?id=1GudAb2jP665CUsCX8q1HWbZGFPeoJpet',
 '/kaggle/working/us_model.keras', quiet=False)

In [ ]:
# ── Extract mammo model if it's a zip ────────────────────────────────────────
import zipfile, os

archive = '/kaggle/working/mammo_model.h5'
extract_dir = '/kaggle/working/mammo_model_extracted'

if zipfile.is_zipfile(archive):
 with zipfile.ZipFile(archive, 'r') as z:
 z.extractall(extract_dir)
 print('Extracted:', os.listdir(extract_dir))
 MAMMO_MODEL_PATH = os.path.join(extract_dir, 'efficientnetv2l_mammography_3class.h5')
else:
 MAMMO_MODEL_PATH = archive

print('Mammo model path:', MAMMO_MODEL_PATH)

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import cv2
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

MAMMO_IMG_ROOT = '/kaggle/input/cbis-ddsm-breast-cancer-image-dataset/jpeg' # 
IMG_SIZE = 456
BATCH_SIZE = 16
SEED = 42
MAMMO_MODEL_PATH = '/kaggle/input/datasets/habibashhefny/mammo-ps-split/base_model.keras' 
OUTPUT_DIR = '/kaggle/working/'

def build_mammo_path(rel_path):
 if rel_path.startswith('CBIS-DDSM'):
 return rel_path.replace('CBIS-DDSM', MAMMO_IMG_ROOT, 1)
 return os.path.join(MAMMO_IMG_ROOT, rel_path)

ft_train_df = pd.read_csv('/kaggle/input/datasets/habibashhefny/mammo-ps-split/mammo_ft_train.csv')
ft_val_df = pd.read_csv('/kaggle/input/datasets/habibashhefny/mammo-ps-split/mammo_ft_val.csv')
final_test_df = pd.read_csv('/kaggle/input/datasets/habibashhefny/mammo-ps-split/mammo_final_test.csv')

ft_train_df['full_img_path'] = ft_train_df['mammo_path'].apply(build_mammo_path)
ft_val_df['full_img_path'] = ft_val_df['mammo_path'].apply(build_mammo_path)
final_test_df['full_img_path'] = final_test_df['mammo_path'].apply(build_mammo_path)

final_test_df.to_csv(os.path.join(OUTPUT_DIR, 'final_test_split.csv'), index=False)

print('Split summary:')
print(f' Fine-tune Train: {len(ft_train_df)} images')
print(f' Fine-tune Val: {len(ft_val_df)} images')
print(f' Final Test: {len(final_test_df)} images')

def load_and_preprocess(img_path, label, augment=False):
 img = cv2.imread(img_path)
 if img is None:
 return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.float32), np.float32(label)
 
 img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
 img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
 img = img.astype(np.float32)
 
 if augment:
 if np.random.rand() > 0.5: img = np.fliplr(img)
 if np.random.rand() > 0.5: img = np.flipud(img)
 img = np.clip(img * np.random.uniform(0.85, 1.15), 0, 255)
 
 return preprocess_input(img), np.float32(label) 

def make_dataset(df, augment=False, shuffle=False):
 paths = df['full_img_path'].values
 labels = df['label'].values.astype(np.float32) 
 
 def load_fn(path, label):
 img, lbl = tf.numpy_function(
 lambda p, l: load_and_preprocess(p.decode(), float(l), augment),
 [path, label], 
 [tf.float32, tf.float32] 
 )
 img.set_shape([IMG_SIZE, IMG_SIZE, 3])
 lbl.set_shape([]) 
 return img, lbl
 
 ds = tf.data.Dataset.from_tensor_slices((paths, labels))
 if shuffle: ds = ds.shuffle(buffer_size=len(df), seed=SEED)
 return ds.map(load_fn, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(ft_train_df, augment=True, shuffle=True)
val_ds = make_dataset(ft_val_df, augment=False, shuffle=False)

print('\nDatasets ready ')

n_b = (ft_train_df['label'] == 0).sum()
n_m = (ft_train_df['label'] == 1).sum()
n = len(ft_train_df)
class_weight = {0: n / (2 * n_b), 1: n / (2 * n_m) * 2}
print('Class weights:', {k: round(v, 3) for k, v in class_weight.items()})

def run_experiment(freeze_until, exp_name, epochs=15):
 print(f'\n{"="*55}')
 print(f' Experiment {exp_name}: freeze_until={freeze_until}')
 print(f'{"="*55}')

 base_model = load_model(MAMMO_MODEL_PATH, compile=False)
 
 inputs = base_model.input
 outputs = base_model.output 
 prob_malignant = outputs[:, 2:3] 
 model = tf.keras.Model(inputs=inputs, outputs=prob_malignant)

 for layer in model.layers: layer.trainable = False
 if freeze_until == 0:
 for layer in model.layers: layer.trainable = True
 else:
 for layer in model.layers[freeze_until:]: layer.trainable = True

 lr = 1e-5 if freeze_until != 0 else 5e-6

 model.compile(
 optimizer=tf.keras.optimizers.Adam(lr),
 loss='binary_crossentropy', 
 metrics=[
 'accuracy', 
 tf.keras.metrics.AUC(name='auc'),
 tf.keras.metrics.Precision(name='precision'),
 tf.keras.metrics.Recall(name='recall')
 ]
 )

 ckpt_path = os.path.join(OUTPUT_DIR, f'mammo_ft_{exp_name}.keras')
 callbacks = [
 ModelCheckpoint(ckpt_path, monitor='val_auc', mode='max', save_best_only=True, verbose=0),
 EarlyStopping(monitor='val_auc', mode='max', patience=5, restore_best_weights=True, verbose=1),
 ReduceLROnPlateau(monitor='val_auc', mode='max', factor=0.5, patience=3, min_lr=1e-8, verbose=0)
 ]

 history = model.fit(
 train_ds, validation_data=val_ds,
 epochs=epochs, callbacks=callbacks,
 class_weight=class_weight, verbose=1
 )

 best_val_auc = max(history.history['val_auc'])
 print(f' Best Val AUC: {best_val_auc:.4f}')

 return model, history, best_val_auc, ckpt_path

results = {}

results['A_last10'], hist_A, auc_A, ckpt_A = run_experiment(-10, 'A_last10')

results['B_last30'], hist_B, auc_B, ckpt_B = run_experiment(-30, 'B_last30')


print('\n' + '='*55)
print('RESULTS SUMMARY')
print('='*55)
print(f' Exp A (last 10 layers): Val AUC = {auc_A:.4f}')
print(f' Exp B (last 30 layers): Val AUC = {auc_B:.4f}')

best_exp = max([('A', auc_A), ('B', auc_B)], key=lambda x: x[1])

print(f'\n Best experiment: {best_exp[0]} with AUC = {best_exp[1]:.4f}')